# Model Pipeline

## Initial data and folder structure.

It is important to ensure that the initial files are organized in a consistent structure.
The original training dataset `dynamic-rhythms-train-data`, provided by the contest, must be placed inside the raw data directory. Additionally, the meteorological data will be generated and stored by us in the same section. 

Here is the initial data expected folder structure (as seen from the root directory of the project):
```

dynamic-rythms/
│
├── data/
│   ├── external/                # Data from third-party sources.
│   ├── interim/                 # Intermediate data files.
│   └── raw/                     # Original, immutable data
│       ├── dynamic-rhythms-train-data/ # Provided by the contest
│       │   └── data/
│       │       ├── eaglei_data/
│       │       └── NOAA_StormEvents/
│       └── meteorological/      # API-generated storm condition data.
```


## Data Gathering.

a) **Outage Information**: We use the outage data. To ensure we focus on impactful events, we define a relevance threshold based on the number of people affected. Specifically, we only consider outages that impacted more than $10^{3.5}$, which corresponds to approximately $3162$ people. This allows us to filter out minor incidents and concentrate on events with significant societal impact. Additionally, we group nearby outage reports that occurred within a defined time frame and affected the same  neighboring counties. This allowed us to treat such cases as a single, prolonged event rather than multiple isolated incidents.

b) **Storm Information**: We organized storm data at an *episode level*, grouping together related storm events that occurred within a single system. For each storm episode, we defined a start and end date to establish its duration. Recognizing that large storms can impact multiple counties simultaneously, we created a composite identifier by combining the storm episode ID with the county FIPS code. This new field, episode_fips_id, uniquely represents the intersection between a specific storm and a specific county, allowing us to track localized impacts of broader storm systems.

c) **County Information**: We obtained official county boundaries from the U.S. Census Bureau using the  shapefile in this [link](https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_5m.zip). This shapefile provides precise geospatial data for all U.S. counties. We use this information to provide our meteorological API requests with accurate geographic points. 

d) **Outage + Storm**: To associate outages with specific storm events, we merged storm and outage data based on temporal proximity. While we acknowledge this is a simplification, it provides a useful approximation for identifying likely storm-related outages. Specifically, we considered an outage to be linked to a storm if it occurred during the storm episode or within one day after its end. This rule allows us to flag storms that likely caused disruptions, and to distinguish them from those that passed without triggering significant outages.

e) **Defining the dataset scope**: At first instance, we narrowed our focus to include only storm episodes that were linked to at least one outage. This filtered set forms the foundation of our training dataset, ensuring that the model is initially trained on events with known impacts. By starting with confirmed storm-outage associations, we simplify the problem space and prioritize learning from patterns with clear outcomes, which will help refine and scale the model in future iterations (which are outside this scope).

f-g) **API Calls**: We then generate a Dataframe containing information of the storms that resulted in at least one outage, merged with the county dataset. Since we have access to both storm and outage durations, we define a temporal window that spans from one day before to one day after each storm. This window allows us to later request hourly meteorological data relevant to each storm's lifecycle. To represent the geographic location of each county, we use its centroid coordinates (latitude and longitude), as pinpointing the exact affected city within a county is not always feasible. With the combination of the `episode_fips_id`, county centroid location and the defined date-time window, we are able to generate the corresponding API calls in batch, retrieving the meteorological conditions that potentially led to the observed outages. API calls are made with the [Power project](https://power.larc.nasa.gov/). The result of each API call is a JSON file containing detailed hourly meteorological conditions within the defined temporal window. Each JSON file is named using the corresponding episode_fips_id, enabling easy traceability and alignment with the original storm-outage pair. These files will later be parsed and transformed into structured features for the machine learning model.




The first step is to join outages and storms. 

In [1]:
import os
import pandas as pd
import numpy as np

import utils_dhm as ut

In [ ]:
YEAR = 